In [1]:
# Install required libraries for Week 2 task

!pip install -q --upgrade google-generativeai
!pip install -q langchain-google-genai
!pip install -q pageindex
!pip install -q rank-bm25
!pip install -q reportlab
!pip install -q nltk

In [2]:
# import required libraries
import os
import re
import json
import nltk
from nltk.corpus import stopwords
import google.generativeai as genai

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# loading gemini api key

from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)
#loads gemini API key from secrets and connect gemini to notebook.

In [4]:
# Load news headlines
import json
with open("news_headlines.json", "r", encoding="utf-8") as f:
    headlines = json.load(f)

print("Total Headlines:", len(headlines))

Total Headlines: 83


In [5]:
for i, headline in enumerate(headlines[:5], 1):
    print(f"{i}. {headline}")
  #checking top headings

1. NEET paper leak: CBI nabs NTA-appointed expert from Pune for leaking physics questions
2. 'If parents are IAS officers, then why children seek quota?': SC's big observation on reservation
3. Pakistan announce ODI squad for Australia series; Babar Azam returns, while Mohammad Rizwan dropped
4. Maa Behen trailer out: One dead body, three women and unlimited chaos; all about Madhuri Dixit film
5. OPINION | Dhurandhar in Pakistan: Terrorists on the run


In [6]:
from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    return " ".join(words)

cleaned_docs = [clean_text(doc) for doc in headlines]

print("Total Documents:", len(cleaned_docs))
print("\nSample Cleaned Headline:")
print(cleaned_docs[0])
# Here I cleaned the headlines dataset.
# I converted all text to lowercase and removed special characters.
# I also removed common stopwords to keep only important words.
# The cleaned text will be used for creating the retrieval corpus.

Total Documents: 83

Sample Cleaned Headline:
neet paper leak cbi nabs ntaappointed expert pune leaking physics questions


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# create corpus chunks
chunks = []
for doc in cleaned_docs:
    chunks.append(doc)
print("Total Chunks:", len(chunks))
print("\nFirst Chunk:")
print(chunks[0])
# each cleaned headline is stored as a separate chunk.
# these chunks will be combined later and used for indexing in PageIndex.
# Keeping headlines as individual chunks helps preserve their information.

Total Chunks: 83

First Chunk:
neet paper leak cbi nabs ntaappointed expert pune leaking physics questions


In [8]:
# load pageindex api key
from pageindex import PageIndexClient
from google.colab import userdata
PAGEINDEX_API_KEY = userdata.get("PAGEINDEX_API_KEY")
client = PageIndexClient(
    api_key=PAGEINDEX_API_KEY)
# This client will be used to upload and retrieve documents from PageIndex.
# The API key is loaded securely from Colab Secrets.

In [9]:
# create pdf from corpus
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
pdf = SimpleDocTemplate("week1_corpus.pdf")
styles = getSampleStyleSheet()
content = []
for chunk in chunks:
    content.append(
        Paragraph(chunk, styles["BodyText"])
    )
pdf.build(content)
# PageIndex accepts PDF files for indexing.
# Therefore, the cleaned corpus is converted into a PDF document.
# each chunk is added as a paragraph inside the PDF.

In [10]:
# this step confirms that the PDF was created successfully.
import os
print(os.path.exists("week1_corpus.pdf"))
print(os.path.getsize("week1_corpus.pdf"))

True
6960


In [11]:
# upload pdf to pageindex
response = client.submit_document(
    file_path="week1_corpus.pdf"
)
DOC_ID = response["doc_id"]
print("Document ID:", DOC_ID)
# the PDF is uploaded to PageIndex for indexing.
# DOC_ID is stored because it will be needed in later retrieval steps.

Document ID: pi-cmpvjhwgr01wb01qu3uwiljd9


In [12]:
# wait for indexing
import time
while not client.is_retrieval_ready(DOC_ID):
    print("Indexing...")
    time.sleep(10)
print("Indexing complete")
# pageIndex needs some time to process the uploaded document.
# this loop checks the indexing status until retrieval becomes available.

Indexing...
Indexing complete


In [13]:
# get indexed content
tree = client.get_tree(DOC_ID)
print("Retrieval Ready:", tree["retrieval_ready"])
print("Status:", tree["status"])
# the document tree contains the indexed nodes created by PageIndex.
# these nodes will be converted into documents for retrieval.

Retrieval Ready: True
Status: completed


In [14]:
# create document list

documents = []

for node in tree["result"]:

    documents.append({

        "content": node["text"],

        "metadata": {
            "title": node["title"],
            "node_id": node["node_id"],
            "page": node["page_index"]
        }

    })

print("Total Documents:", len(documents))

# Each node from the PageIndex tree is converted into a document. Metadata is stored along with the text content for easy reference.

Total Documents: 3


In [15]:
print(documents[0]["content"][:300])

neet paper leak cbi nabs ntaappointed expert pune leaking physics questions

parents ias officers children seek quota scs big observation reservation

pakistan announce odi squad australia series babar azam returns mohammad rizwan dropped

maa behen trailer one dead body three women unlimited chaos 


In [16]:
# create bm25 index
from rank_bm25 import BM25Okapi
tokenized_corpus = []
for doc in documents:
    tokenized_corpus.append(
        doc["content"].split()
    )
bm25 = BM25Okapi(tokenized_corpus)
# BM25 is used to rank documents based on keyword matching.
# Each document is tokenized into words before creating the index.

In [17]:
# retrieval function
def retrieve(query, k=3):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    results = []

    for idx in ranked_indices[:k]:
        results.append(documents[idx])

    return results
# this function finds the most relevant documents for a given user query using BM25 ranking.

In [18]:
# test retrieval
results = retrieve("neet paper leak")
for i, doc in enumerate(results, 1):
    print(f"\nResult {i}")
    print(doc["content"][:200])
# testing retrieval before connecting it to Gemini.
# this helps verify that BM25 is returning relevant documents.


Result 1
neet paper leak cbi nabs ntaappointed expert pune leaking physics questions

parents ias officers children seek quota scs big observation reservation

pakistan announce odi squad australia series baba

Result 2
akshay kumar donates rs 1 crore remote jammu kashmir border school

exclusive sonali kulkarni says wasnt first choice playing hrithiks mother mission

yoga swami ramdev strengthen bones muscles natura

Result 3
heart develop cancer doctor reveals surprising medical fact

heat stroke symptoms doctor explains visit doctor heat stroke

budh gochar 2026 mercurys move gemini may bring career growth financial gain


In [19]:
model = genai.GenerativeModel(
    "gemini-2.5-flash"
)
print("Gemini model ready")
# gemini will generate answers using the retrieved context from BM25.

Gemini model ready


In [20]:
# rag function
def ask_rag(query):

    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        [doc["content"] for doc in retrieved_docs]
    )

    prompt = f"""
Context:
{context}

Question:
{query}

Answer using only the given context.
"""

    response = model.generate_content(prompt)

    return response.text

# this function combines retrieval and generation.
# relevant documents are retrieved first and then passed to Gemini to generate the final answer.

In [21]:
# test rag system
question = "What happened in the NEET paper leak case?"

answer = ask_rag(question)

print(answer)


In the NEET paper leak case, the CBI nabbed an NTA-appointed expert in Pune for leaking physics questions.


In [22]:
print(ask_rag("Why are children seeking quota according to the news?"))

According to the news, children of IAS officers are seeking a quota related to SCS (Scheduled Castes/Tribes) and reservation. The specific reason "why" they are seeking it is not elaborated in the provided context.


In this task, I build a simple RAG system using news headlines data.
First the news headlines were loaded and cleaned by removing unwanted characters and stopwords. The cleaned data was then converted into a corpus and upload to Pageindex for indexing.
After indexing the document content was extracted and stored in a format suitable for retrieval. BM25 was used as the retrieval method to find the most relevant news articles for a user query.
Finally Gemini was used to generate answers based only on the retrieved context.
The system was tested with different questions and was able to return relevant answers from the news dataset. This shows how retrieval and generation can be combined to build a simple question-answering system.